In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from joblib import Parallel, delayed

root_path = Path("/mnt/cluster/datasets/threading_il/v4/")
episode_dirs = [d for d in root_path.iterdir() if d.is_dir()]
datasets_paths = [str(d / 'episode.csv') for d in episode_dirs]

def process_episode(path):
    episode_df = pd.read_csv(path)
    depth_paths = episode_df['frameLeftPath'].apply(lambda x: x.replace("framesLeft", "stereoDepth").replace(".png", ".tiff")).tolist()
    local_min_disp = np.inf
    local_max_disp = -np.inf
    for disparity_path in depth_paths:
        disparity_image = cv2.imread(disparity_path, cv2.IMREAD_UNCHANGED)
        if disparity_image is None:
            continue
        min_d = disparity_image.min()
        max_d = disparity_image.max()
        if min_d < local_min_disp:
            local_min_disp = min_d
        if max_d > local_max_disp:
            local_max_disp = max_d
    return local_min_disp, local_max_disp

results = Parallel(n_jobs=-1)(delayed(process_episode)(path) for path in datasets_paths)

global_min_disp = min(r[0] for r in results if r[0] != np.inf)
global_max_disp = max(r[1] for r in results if r[1] != -np.inf)

MIN_DEPTH = 1.0 / global_max_disp
MAX_DEPTH = 1.0 / global_min_disp

print(f"Global min depth: {MIN_DEPTH}")
print(f"Global max depth: {MAX_DEPTH}")

Global min depth: 0.007743687900641262
Global max depth: 0.1426304120135479
